# Exploratory Data Analysis in SQL

**Platform:** DataCamp  
**Level:** Intermediate  
**Duration:** 4 hours | 16 videos | 57 exercises  
**Database:** PostgreSQL  
**Datasets:** Stack Overflow, Fortune 500, Evanston 311 help requests

---

**Course Description:**  
Learn how to explore a SQL database — the tables, relationships between them, and the data stored in them.  
Topics: numeric, character, and date/time data types; aggregation; cleaning messy data; temp tables.

---

## Table of Contents
1. [Chapter 1: What's in the Database?](#chapter-1)
2. [Chapter 2: Summarizing and Aggregating Numeric Data](#chapter-2)
3. [Chapter 3: Exploring Categorical Data and Unstructured Text](#chapter-3)
4. [Chapter 4: Working with Dates and Timestamps](#chapter-4)

---
## 🛠️ Local Setup — DuckDB

Run the cell below **once** before working on any exercise.  
It loads all course datasets into an in-memory DuckDB database.

**Data files required** (gitignored — download from DataCamp Resources panel):
```
data/01-eda-sql/evanston311.csv
data/01-eda-sql/fortune500.csv
data/01-eda-sql/stackoverflow.csv
```
Schema reference: `data/01-eda-sql/create-database.sql`

In [ ]:
import duckdb
from pathlib import Path

# Resolve absolute path to data dir (works regardless of cwd)
NOTEBOOK_DIR = Path(__file__).parent if "__file__" in dir() else Path().resolve()
DATA_DIR = NOTEBOOK_DIR / "data" / "01-eda-sql"

# Connect to in-memory DuckDB
con = duckdb.connect()

# --- Tables from CSV (large datasets) ---
con.execute(f"""
    CREATE TABLE fortune500 AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/fortune500.csv', nullstr='NA');
""")
con.execute(f"""
    CREATE TABLE evanston311 AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/evanston311.csv', nullstr='NA');
""")
con.execute(f"""
    CREATE TABLE stackoverflow AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/stackoverflow.csv', nullstr='NA');
""")

# --- Tables from create-database.sql (small lookup tables with INSERT data) ---
con.execute("""
    CREATE TABLE company (
        id INT,
        exchange VARCHAR,
        ticker VARCHAR,
        name VARCHAR,
        parent_id INT
    );
    INSERT INTO company VALUES
    (1, 'nasdaq', 'PYPL',  'PayPal Holdings Incorporated', NULL),
    (2, 'nasdaq', 'AMZN',  'Amazon.com Inc', NULL),
    (3, 'nasdaq', 'MSFT',  'Microsoft Corp.', NULL),
    (4, 'nasdaq', 'MDB',   'MongoDB', NULL),
    (5, 'nasdaq', 'DBX',   'Dropbox', NULL),
    (6, 'nasdaq', 'AAPL',  'Apple Incorporated', NULL),
    (7, 'nasdaq', 'CTXS',  'Citrix Systems', NULL),
    (8, 'nasdaq', 'GOOGL', 'Alphabet', NULL),
    (9, 'nyse',   'IBM',   'International Business Machines Corporation', NULL),
    (10,'nasdaq', 'ADBE',  'Adobe Systems Incorporated', NULL),
    (11, NULL,    NULL,    'Stripe', NULL),
    (12, NULL,    NULL,    'Amazon Web Services', 2),
    (13, NULL,    NULL,    'Google LLC', 8),
    (14,'nasdaq', 'EBAY',  'eBay, Inc.', NULL);
""")

con.execute("""
    CREATE TABLE tag_company (
        tag VARCHAR,
        company_id INT
    );
    INSERT INTO tag_company (tag, company_id) VALUES
    ('actionscript', 10), ('actionscript-3', 10),
    ('amazon', 2), ('amazon-api', 2), ('amazon-appstore', 2),
    ('amazon-cloudformation', 12), ('amazon-cloudfront', 12),
    ('amazon-cloudsearch', 12), ('amazon-cloudwatch', 12),
    ('amazon-cognito', 12), ('amazon-data-pipeline', 12),
    ('amazon-dynamodb', 12), ('amazon-ebs', 12), ('amazon-ec2', 12),
    ('amazon-ecs', 12), ('amazon-elastic-beanstalk', 12),
    ('amazon-elasticache', 12), ('amazon-elb', 12), ('amazon-emr', 12),
    ('amazon-fire-tv', 2), ('amazon-glacier', 12), ('amazon-kinesis', 12),
    ('amazon-lambda', 12), ('amazon-mws', 12), ('amazon-rds', 12),
    ('amazon-rds-aurora', 12), ('amazon-redshift', 12),
    ('amazon-route53', 12), ('amazon-s3', 12), ('amazon-ses', 12),
    ('amazon-simpledb', 12), ('amazon-sns', 12), ('amazon-sqs', 12),
    ('amazon-swf', 12), ('amazon-vpc', 12), ('amazon-web-services', 12),
    ('android', 13), ('android-pay', 13),
    ('applepay', 6), ('applepayjs', 6),
    ('azure', 3), ('citrix', 7), ('cognos', 9),
    ('dropbox', 5), ('dropbox-api', 5),
    ('excel', 3), ('google-spreadsheet', 13),
    ('ios', 6), ('ios8', 6), ('ios9', 6),
    ('mongodb', 4), ('osx', 6), ('paypal', 1),
    ('sql-server', 3), ('stripe-payments', 11), ('windows', 3);
""")

con.execute("""
    CREATE TABLE tag_type (
        id INTEGER,
        tag VARCHAR,
        type VARCHAR
    );
    INSERT INTO tag_type (tag, type) VALUES
    ('amazon-cloudformation','cloud'), ('amazon-cloudfront','cloud'),
    ('amazon-cloudsearch','cloud'), ('amazon-cloudwatch','cloud'),
    ('amazon-cognito','cloud'), ('amazon-cognito','identity'),
    ('amazon-data-pipeline','cloud'), ('amazon-dynamodb','cloud'),
    ('amazon-dynamodb','database'), ('amazon-ebs','cloud'),
    ('amazon-ec2','cloud'), ('amazon-ecs','cloud'),
    ('amazon-elastic-beanstalk','cloud'), ('amazon-elasticache','cloud'),
    ('amazon-elb','cloud'), ('amazon-emr','cloud'),
    ('amazon-glacier','cloud'), ('amazon-glacier','storage'),
    ('amazon-kinesis','cloud'), ('amazon-lambda','cloud'),
    ('amazon-mws','api'), ('amazon-rds-aurora','cloud'),
    ('amazon-rds','cloud'), ('amazon-rds-aurora','database'),
    ('amazon-rds','database'), ('amazon-redshift','cloud'),
    ('amazon-route53','cloud'), ('amazon-s3','cloud'),
    ('amazon-ses','cloud'), ('amazon-simpledb','cloud'),
    ('amazon-simpledb','database'), ('amazon-sns','cloud'),
    ('amazon-sqs','cloud'), ('amazon-swf','cloud'),
    ('amazon-vpc','cloud'), ('amazon-web-services','cloud'),
    ('amazon','company'), ('android-pay','payment'),
    ('android','mobile-os'), ('applepay','payment'),
    ('applepayjs','payment'), ('azure','cloud'),
    ('citrix','company'), ('dropbox-api','api'),
    ('dropbox','storage'), ('dropbox','cloud'), ('dropbox','company'),
    ('excel','spreadsheet'), ('google-spreadsheet','spreadsheet'),
    ('ios','mobile-os'), ('ios8','mobile-os'), ('ios9','mobile-os'),
    ('mongodb','database'), ('osx','os'),
    ('paypal','payment'), ('paypal','company'),
    ('sql-server','database'), ('stripe-payments','payment'),
    ('windows','os');
""")

# Verify all 6 tables
for table in ['fortune500', 'evanston311', 'stackoverflow', 'company', 'tag_company', 'tag_type']:
    count = con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"✅ {table}: {count:,} rows")

print(f"\nData dir: {DATA_DIR}")
print("DuckDB ready. Use: con.execute(sql).df()")

✅ fortune500: 500 rows
✅ evanston311: 36,431 rows
✅ stackoverflow: 45,238 rows
✅ company: 14 rows
✅ tag_company: 56 rows
✅ tag_type: 59 rows

Data dir: /Users/zeal.v/Desktop/vs_code/data-brain-gym/datacamp/sql-for-business-analysts/data/01-eda-sql
DuckDB ready. Use: con.execute(sql).df()


---
# Chapter 1: What's in the Database?

Start exploring a database by identifying the tables and the foreign keys that link them.  
Look for missing values, count the number of observations, and join tables to understand how they're related.  
Learn about coalescing and casting data along the way.

## Exercise 1 — What's in the database? *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/whats-in-the-database?ex=1) *(requires login)*

**What this video covers:**
When you encounter a new database, the first step is always exploration — identify what tables exist, how many rows each has, and how the tables relate to each other. This video introduces the systematic approach to database exploration that the rest of Chapter 1 builds on.

## Exercise 2 — Explore table sizes *(35 XP)*

**Tables:** `stackoverflow`, `company`, `tag_company`, `tag_type`, `fortune500`

Let's start by exploring five related tables:
- `stackoverflow` — questions asked on Stack Overflow with certain tags
- `company` — information on companies related to tags in `stackoverflow`
- `tag_company` — links `stackoverflow` to `company`
- `tag_type` — type categories applied to tags in `stackoverflow`
- `fortune500` — information on top US companies

Count the number of columns in a table by selecting a few rows and manually counting the columns in the result.

**Which table has the most rows?**

---

**Instructions:**

Query each table with `SELECT COUNT(*)` to count all rows and see which has the most.

---

**Note (local setup):**  
Of the 5 DataCamp tables for this exercise, only 2 are available locally as CSVs: `fortune500` and `stackoverflow`.  
`company`, `tag_company`, and `tag_type` exist only in DataCamp's PostgreSQL environment — no CSV download available.  
`evanston311` is also available locally, but it belongs to a different part of the course — it is not one of these 5 tables.  
We query all 3 local tables. `stackoverflow` is the largest — which matches the correct answer on DataCamp.

---

**Hint:**
- Use `SELECT COUNT(*)` on each table to count all rows.
- The table with the highest count is the answer.

In [7]:
# Exercise 2 — Explore table sizes
# Count rows in each locally available table to find which has the most

sql_evanston = "SELECT count(*) AS row_count FROM evanston311;"
sql_fortune = "SELECT count(*) AS row_count FROM fortune500;"
sql_stackoverflow = "SELECT count(*) AS row_count FROM stackoverflow;"

print("evanston311:")
display(con.execute(sql_evanston).df())

print("fortune500:")
display(con.execute(sql_fortune).df())

print("stackoverflow:")
display(con.execute(sql_stackoverflow).df())

# Answer: stackoverflow has the most rows

evanston311:


,row_count
0,36431


fortune500:


,row_count
0,500


stackoverflow:


,row_count
0,45238


## Exercise 3 — Count missing values *(100 XP)*

**Table:** `fortune500`

Which column of `fortune500` has the most missing values?  
To find out, you'll need to check each column individually — here we check just two: `ticker` and `industry`.

> **Course Note:** If a query takes more than a few seconds, your session may expire.  
> If that happens, it's an error in your query — not a server issue.

---

**Instructions:**

1. Subtract the count of the non-null `ticker` values from the total number of rows in `fortune500`; alias the difference as `missing`.
2. Repeat for the `industry` column: subtract the count of the non-null `industry` values from the total number of rows in `fortune500`; alias the difference as `missing`.

---

**Hint:**
- `count(column_name)` returns the number of **non-null** values in that column.
- `count(*)` counts **all rows**, including NULLs.
- `count(*) - count(column)` = number of NULLs in that column.
- Use `AS missing` to alias the result.

In [ ]:
# Exercise 3 — Count missing values
# Table: fortune500

# Instruction 1: count missing values in the ticker column
sql_1 = """
SELECT count(*) - count(ticker) AS missing
  FROM fortune500;
"""

# Instruction 2: count missing values in the industry column
sql_2 = """
SELECT count(*) - count(industry) AS missing
  FROM fortune500;
"""

print("Missing ticker values:")
display(con.execute(sql_1).df())

print("Missing industry values:")
display(con.execute(sql_2).df())


Missing ticker values:


,missing
0,32


Missing industry values:


,missing
0,13


## Exercise 4 — Join tables *(0 XP)*

**Tables:** `company`, `fortune500`

---

### 🎯 Goal
Find a column that uniquely identifies a company in both `company` and `fortune500`, and use it to join the two tables with an `INNER JOIN`.

### 🔧 Approach
**Pattern:** `INNER JOIN ... ON table1.column = table2.column`

The two tables don't have a formal foreign key relationship in the schema. We need to inspect both tables and find a column where:
1. The column exists in both tables
2. The values are consistent (same format, same data)
3. The column uniquely identifies each company

Both tables have a `name` column — but company names are often written differently (`Amazon.com Inc` vs `Amazon`). Not reliable.
Both tables have a `ticker` column — stock tickers are standardised identifiers (e.g. `AAPL`, `MSFT`). Reliable.

**JOIN column:** `ticker`

---

**Instructions:**
1. Closely inspect `company` and `fortune500` to find a column present in both that can uniquely identify each company.
2. Join `company` and `fortune500` with an `INNER JOIN` on that column.

---

**Hint:**
- `company` and `fortune500` both have `name` columns, but the names aren't consistent — this is not a good join column.
- When a column has the same name in two tables, qualify it with the table name in the `ON` clause: `table1.column = table2.column`.
- `ticker` is a stock exchange symbol — standardised, unique per company, consistent across both tables.

---

**Why this matters:**
An `INNER JOIN` returns only rows that have a matching value in both tables. Choosing the wrong column to join on produces either zero rows (no matches) or incorrect duplicates. Always verify your join key is consistent and unique before using it.

In [10]:
# Exercise 4 — Join tables
# Tables: company, fortune500
# Goal: join the two tables on the column that uniquely identifies each company

sql = """
SELECT company.name        -- select company name from the company table
  FROM company
       INNER JOIN fortune500        -- join to fortune500
       ON company.ticker = fortune500.ticker;  -- ticker is the consistent unique identifier
"""

# Expected: 8 rows — only companies that appear in both tables
display(con.execute(sql).df())

,name
0,Apple Incorporated
1,Amazon.com Inc
2,Alphabet
3,Microsoft Corp.
4,International Business Machines Corporation
5,PayPal Holdings Incorporated
6,"eBay, Inc."
7,Adobe Systems Incorporated


## Exercise 5 — The keys to the database *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/whats-in-the-database?ex=5) *(requires login)*

**What this video covers:**
Primary keys uniquely identify each row in a table. Foreign keys reference a primary key in another table, enforcing referential integrity — a row in a child table cannot reference a value that doesn't exist in the parent. Understanding the key structure of a database is the foundation for writing correct JOINs.

## Exercise 6 — Foreign keys *(35 XP)*

**Tables:** `tag_type`, `stackoverflow`

---

### 🎯 Goal
Understand why the `tag` column in `tag_type` **cannot** be a foreign key that references the `tag` column in `stackoverflow`.

### 🔧 Approach
**Concept:** A foreign key must reference a column that is a **primary key** (or has a unique constraint) in the referenced table. That means every value in the referenced column must be unique — no duplicates allowed.

We check: does `stackoverflow.tag` contain duplicate values?

---

**Question:** Using what you know about foreign keys, why can't the `tag` column in `tag_type` be a foreign key referencing `tag` in `stackoverflow`?

**Possible answers:**
- `stackoverflow.tag` is not a primary key
- `tag_type.tag` contains NULL values
- ✅ **`stackoverflow.tag` contains duplicate values** ← correct
- `tag_type.tag` does not contain all the values in `stackoverflow.tag`

---

**Why this matters:**
A foreign key enforces referential integrity — it guarantees that every value in the child column exists in the parent column. But the parent column must itself be unique (primary key or unique constraint). `stackoverflow` tracks questions **per tag per date**, so the same tag appears on many rows. It cannot serve as a parent for a foreign key.

---

**Hint:**
Examine `stackoverflow.tag` to look for reasons why it doesn't uniquely identify rows.

In [11]:
# Exercise 6 — Foreign keys
# Verify: does stackoverflow.tag contain duplicate values?
# A foreign key parent column must have unique values — if it has duplicates, it cannot be a FK target.

sql = """
SELECT tag,
       count(*) AS occurrences   -- count how many rows each tag appears on
  FROM stackoverflow
 GROUP BY tag
 ORDER BY occurrences DESC
 LIMIT 5;                        -- show the top 5 most repeated tags
"""

print("Top 5 most repeated tags in stackoverflow:")
display(con.execute(sql).df())

# ✅ Answer: stackoverflow.tag contains duplicate values — same tag appears on many rows (one per date).
# Therefore it cannot be a primary key, and tag_type.tag cannot reference it as a foreign key.

Top 5 most repeated tags in stackoverflow:


,tag,occurrences
0,google-spreadsheet,987
1,ios9,987
2,actionscript,987
3,ios8,987
4,actionscript-3,987


## Exercise 7 — Read an entity relationship diagram *(100 XP)*

**Tables:** `tag_type`, `tag_company`, `company`

---

### 🎯 Goal
Find the most common `stackoverflow` tag type, then identify which companies have tags of that type — by joining three tables using the entity relationship diagram.

### 🔧 Approach
**Two-step pattern:** aggregate first to find the answer → use that answer to filter a multi-table JOIN.

**Step 1:** `tag_type` → `GROUP BY type` + `COUNT(tag)` → find most common type  
**Step 2:** `company` → `LEFT JOIN tag_company` (on `id = company_id`) → `LEFT JOIN tag_type` (on `tag = tag`) → `WHERE type = 'cloud'`

The ERD tells you which column to join on between each pair of tables:
- `company.id` ↔ `tag_company.company_id`
- `tag_company.tag` ↔ `tag_type.tag`

---

**Instructions 1/2 (50 XP):**
- Using the `tag_type` table, count the number of tags with each `type`.
- Order the results to find the most common tag `type`.

**Instructions 2/2 (50 XP):**
- Join `tag_company`, `company`, and `tag_type`, keeping only mutually occurring records.
- Select `company.name`, `tag_type.tag`, and `tag_type.type` for tags with the most common type from step 1.

---

**Hint:**
- Step 1: `SELECT type, COUNT(tag) AS count FROM tag_type GROUP BY type ORDER BY count DESC`
- Step 2: Start from `company`, LEFT JOIN to `tag_company` on `company.id = tag_company.company_id`, then LEFT JOIN `tag_type` on `tag_company.tag = tag_type.tag`
- Filter with `WHERE type = 'cloud'` — the answer from step 1

---

**Why this matters:**
Real-world questions rarely live in a single table. The ERD is your map — it shows exactly which columns to join on at each step. The two-step pattern (aggregate to find the answer → use the answer in a filter) is one of the most common analytical patterns in SQL.


In [ ]:
# Exercise 7 — Instruction 1/2: count tags per type in tag_type
# Goal: find the most common stackoverflow tag type

sql_1 = """
SELECT type,
       COUNT(tag) AS count     -- count number of tags with each type
  FROM tag_type
 GROUP BY type
 ORDER BY count DESC;          -- most common type first
"""

print("Tag types by frequency:")
display(con.execute(sql_1).df())

# ✅ Most common type: cloud (31 tags)


In [ ]:
# Exercise 7 — Instruction 2/2: three-table JOIN to find companies with cloud tags
# Join company → tag_company → tag_type, filter to most common type from step 1

sql_2 = """
SELECT company.name,           -- company name
       tag_type.tag,           -- the tag itself
       tag_type.type           -- the tag type (will all be 'cloud')
  FROM company
       LEFT JOIN tag_company                              -- join to tag_company bridge table
       ON company.id = tag_company.company_id            -- company.id matches tag_company.company_id
       LEFT JOIN tag_type                                -- join to tag_type
       ON tag_company.tag = tag_type.tag                 -- tag_company.tag matches tag_type.tag
 WHERE tag_type.type = 'cloud';                          -- filter to most common type from step 1
"""

print("Companies with cloud tags:")
display(con.execute(sql_2).df())


## Exercise 8 — Coalesce *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/whats-in-the-database?ex=8) *(requires login)*

**What this video covers:**
`COALESCE(a, b, c)` returns the first non-NULL value from its arguments. It is the standard SQL tool for substituting a default value when a column contains NULLs — essential before aggregations or JOINs where NULLs would otherwise distort results or drop rows.

## Exercise 9 — Column types and constraints *(50 XP)*

In [ ]:
# Exercise 9 — Column types and constraints

## Exercise 10 — Effects of casting *(100 XP)*

In [ ]:
# Exercise 10 — Effects of casting

## Exercise 11 — Summarize the distribution of numeric values *(100 XP)*

In [ ]:
# Exercise 11 — Summarize the distribution of numeric values

---
# Chapter 2: Summarizing and Aggregating Numeric Data

Build on your aggregation skills by exploring numeric data in more depth.  
Use functions to summarize distributions, detect anomalies, compute correlations,  
and organize your work with temporary tables.

## Exercise 1 — Numeric data types and summary functions *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/summarizing-and-aggregating-numeric-data?ex=1) *(requires login)*

**What this video covers:**
SQL distinguishes between integer, numeric, and floating point types — each with different precision and behaviour under division. Summary functions (`min`, `max`, `avg`, `sum`, `stddev`, `variance`) are introduced as the primary tools for describing the distribution of a numeric column.

## Exercise 2 — Division *(50 XP)*

In [ ]:
# Exercise 2 — Division

## Exercise 3 — Explore with division *(100 XP)*

In [ ]:
# Exercise 3 — Explore with division

## Exercise 4 — Summarize numeric columns *(100 XP)*

In [ ]:
# Exercise 4 — Summarize numeric columns

## Exercise 5 — Summarize group statistics *(100 XP)*

In [ ]:
# Exercise 5 — Summarize group statistics

## Exercise 6 — Exploring distributions *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/summarizing-and-aggregating-numeric-data?ex=6) *(requires login)*

**What this video covers:**
`trunc()` truncates a number to a specified number of decimal places (or to the nearest integer), enabling grouping of continuous values into bins. Combined with `generate_series()`, this creates histograms directly in SQL — a powerful pattern for exploring value distributions without needing a BI tool.

## Exercise 7 — Truncate *(100 XP)*

In [ ]:
# Exercise 7 — Truncate

## Exercise 8 — Generate series *(100 XP)*

In [ ]:
# Exercise 8 — Generate series

## Exercise 9 — More summary functions *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/summarizing-and-aggregating-numeric-data?ex=9) *(requires login)*

**What this video covers:**
`corr(x, y)` measures the linear correlation between two columns (-1 to 1). `percentile_disc(p)` and `percentile_cont(p)` return the p-th percentile of a distribution — `percentile_disc` returns an actual value from the data, `percentile_cont` interpolates. Together these functions reveal relationships and spread that `avg` alone cannot show.

## Exercise 10 — Correlation *(100 XP)*

In [ ]:
# Exercise 10 — Correlation

## Exercise 11 — Mean and Median *(100 XP)*

In [ ]:
# Exercise 11 — Mean and Median

## Exercise 12 — Creating temporary tables *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/summarizing-and-aggregating-numeric-data?ex=12) *(requires login)*

**What this video covers:**
Temporary tables (`CREATE TEMP TABLE`) exist only for the duration of a session and are invisible to other users. They simplify complex multi-step queries by storing intermediate results — instead of nesting subqueries, you break the work into readable steps. The pattern: create temp table → query it → drop or let it expire.

## Exercise 13 — Create a temp table *(100 XP)*

In [ ]:
# Exercise 13 — Create a temp table

## Exercise 14 — Create a temp table to simplify a query *(100 XP)*

In [ ]:
# Exercise 14 — Create a temp table to simplify a query

## Exercise 15 — Insert into a temp table *(100 XP)*

In [ ]:
# Exercise 15 — Insert into a temp table

---
# Chapter 3: Exploring Categorical Data and Unstructured Text

Clean and summarize categorical and unstructured text using string operations,  
pattern matching, grouping, recoding values, and creating indicator variables.

## Exercise 1 — Character data types and common issues *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/exploring-categorical-data-and-unstructured-text?ex=1) *(requires login)*

**What this video covers:**
Character types (`char`, `varchar`, `text`) differ in storage and padding behaviour. Common problems in real-world data: inconsistent casing (`Apple` vs `apple`), leading/trailing whitespace, and mixed formats. `lower()`, `upper()`, `trim()`, and `length()` are the primary tools for detecting and cleaning these issues.

## Exercise 2 — Count the categories *(100 XP)*

In [ ]:
# Exercise 2 — Count the categories

## Exercise 3 — Spotting character data problems *(100 XP)*

In [ ]:
# Exercise 3 — Spotting character data problems

## Exercise 4 — Cases and spaces *(100 XP)*

In [ ]:
# Exercise 4 — Cases and spaces

## Exercise 5 — Trimming *(100 XP)*

In [ ]:
# Exercise 5 — Trimming

## Exercise 6 — Exploring unstructured text *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/exploring-categorical-data-and-unstructured-text?ex=6) *(requires login)*

**What this video covers:**
Free-text fields (like service request descriptions) contain valuable information but require pattern matching to extract it. `LIKE` and `ILIKE` match patterns with `%` (any sequence) and `_` (any single character). This enables categorising, filtering, and counting records based on keywords in unstructured text.

## Exercise 7 — Splitting and concatenating text *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/exploring-categorical-data-and-unstructured-text?ex=7) *(requires login)*

**What this video covers:**
`split_part(string, delimiter, position)` extracts a specific part of a delimited string (e.g. extract the domain from an email address). `||` or `concat()` joins strings together. `left(string, n)` and `right(string, n)` extract a fixed number of characters from either end — useful for truncating long values or extracting codes.

## Exercise 8 — Concatenate strings *(100 XP)*

In [ ]:
# Exercise 8 — Concatenate strings

## Exercise 9 — Split strings on a delimiter *(100 XP)*

In [ ]:
# Exercise 9 — Split strings on a delimiter

## Exercise 10 — Shorten long strings *(100 XP)*

In [ ]:
# Exercise 10 — Shorten long strings

## Exercise 11 — Strategies for multiple transformations *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/exploring-categorical-data-and-unstructured-text?ex=11) *(requires login)*

**What this video covers:**
When a column needs multiple cleaning steps, `CASE WHEN` allows recoding values into new categories (e.g. grouping many specific categories into broader buckets like `'Other'`). This is the SQL equivalent of a lookup table or a mapping function — essential for standardising messy categorical data before analysis.

## Exercise 12 — Create an "other" category *(100 XP)*

In [ ]:
# Exercise 12 — Create an "other" category

## Exercise 13 — Group and recode values *(100 XP)*

In [ ]:
# Exercise 13 — Group and recode values

## Exercise 14 — Create a table with indicator variables *(100 XP)*

In [ ]:
# Exercise 14 — Create a table with indicator variables

---
# Chapter 4: Working with Dates and Timestamps

Assess date and timestamp fields through extraction, truncation, interval arithmetic,  
and `generate_series` to construct comprehensive temporal analyses.

## Exercise 1 — Date/time types and formats *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/working-with-dates-and-timestamps?ex=1) *(requires login)*

**What this video covers:**
SQL date/time types: `date` (date only), `time`, `timestamp` (date + time), `timestamptz` (with timezone). ISO 8601 is the standard format: `YYYY-MM-DD HH:MM:SS`. Storing dates as proper date types — not strings — enables arithmetic, comparison, and extraction functions that strings cannot support.

## Exercise 2 — ISO 8601 *(50 XP)*

In [ ]:
# Exercise 2 — ISO 8601

## Exercise 3 — Date comparisons *(100 XP)*

In [ ]:
# Exercise 3 — Date comparisons

## Exercise 4 — Date arithmetic *(100 XP)*

In [ ]:
# Exercise 4 — Date arithmetic

## Exercise 5 — Completion time by category *(100 XP)*

In [ ]:
# Exercise 5 — Completion time by category

## Exercise 6 — Date/time components and aggregation *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/working-with-dates-and-timestamps?ex=6) *(requires login)*

**What this video covers:**
`date_part('field', timestamp)` and `extract(field FROM timestamp)` pull out individual components: year, month, day, hour, dow (day of week, 0=Sunday). `date_trunc('precision', timestamp)` rounds a timestamp down to the nearest unit (month, week, day) — the key tool for grouping time series data into periods.

## Exercise 7 — Date parts *(100 XP)*

In [ ]:
# Exercise 7 — Date parts

## Exercise 8 — Variation by day of week *(100 XP)*

In [ ]:
# Exercise 8 — Variation by day of week

## Exercise 9 — Date truncation *(100 XP)*

In [ ]:
# Exercise 9 — Date truncation

## Exercise 10 — Aggregating with date/time series *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/working-with-dates-and-timestamps?ex=10) *(requires login)*

**What this video covers:**
`generate_series(start, end, interval)` generates a sequence of dates or timestamps — useful for creating a complete date spine that includes periods with no data. When LEFT JOINed with actual data, it surfaces missing dates as rows with NULL counts, making gaps in time series visible and explicit.

## Exercise 11 — Find missing dates *(100 XP)*

In [ ]:
# Exercise 11 — Find missing dates

## Exercise 12 — Custom aggregation periods *(100 XP)*

In [ ]:
# Exercise 12 — Custom aggregation periods

## Exercise 13 — Monthly average with missing dates *(100 XP)*

In [ ]:
# Exercise 13 — Monthly average with missing dates

## Exercise 14 — Time between events *(Video / 50 XP)*

> 🎬 [Watch on DataCamp](https://campus.datacamp.com/courses/exploratory-data-analysis-in-sql/working-with-dates-and-timestamps?ex=14) *(requires login)*

**What this video covers:**
`lag(column) OVER (ORDER BY ...)` is a window function that returns the value from the previous row — enabling calculation of the gap between consecutive events. Subtracting two timestamps gives an `interval` type; `date_part('epoch', interval)` converts it to seconds for numeric comparison. This is the foundation of event gap and response time analysis.

## Exercise 15 — Longest gap *(100 XP)*

In [ ]:
# Exercise 15 — Longest gap

## Exercise 16 — Rats! *(100 XP)*

In [ ]:
# Exercise 16 — Rats!

## Exercise 17 — Wrap-up *(50 XP)*

In [ ]:
# Exercise 17 — Wrap-up